# 🔍 04 - Seaborn: Relationships and Categories

Notebook 03 drew the shape of **one** variable at a time. This notebook draws how variables relate: two numeric columns (scatter plots and trend lines), a numeric column across categories (bar and point plots), and whole tables at once (heatmaps and pair plots).

Two promises from Module 07 get paid off here: the penguin **Simpson's paradox** from the lab, and the Anscombe datasets with a straight line fitted through them.

## ✅ Learning Goals
- Draw scatter plots with `hue`, `style`, and `size`, and handle overplotting with transparency
- Add trend lines with `sns.regplot()` / `sns.lmplot()`, and see why one line can mislead
- Show Simpson's paradox in a single chart
- Choose between `countplot`, `barplot`, and `pointplot`, and explain what Seaborn's error bars mean
- Turn a correlation matrix and a pivot table into heatmaps, with the right color scale for each
- Survey every pair of variables with `sns.pairplot()`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from pathlib import Path

OUT = Path('04-OutputFiles')
OUT.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
SPECIES_ORDER = ['Adelie', 'Chinstrap', 'Gentoo']

penguins = pd.read_csv('penguins.csv').dropna(subset=MEASUREMENTS)
penguins['sex'] = penguins['sex'].str.title()

titanic = pd.read_csv('titanic.csv')
titanic['class'] = titanic['Pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})

## 1. Scatter Plots with `sns.scatterplot()`
Each point is one row. Extra columns can control the point's **color** (`hue`), **marker shape** (`style`), and **size** (`size`):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=penguins, x='flipper_length_mm', y='body_mass_g',
                hue='species', style='sex', hue_order=SPECIES_ORDER, ax=ax)
ax.set(title='Flipper Length vs. Body Mass', xlabel='Flipper length (mm)', ylabel='Body mass (g)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.show()

This is Module 07's strongest correlation (r ≈ 0.87), and now you can see its structure: three clusters (species), with males heavier within each one.

⚠️ Each extra encoding is one more thing for the reader to decode. `hue` + `style` is about the limit. Beyond that, use panels (`sns.relplot(..., col=)`, the figure-level version of `scatterplot`).

### Overplotting
When many points land in the same place, they hide each other. Titanic `Age` vs. `Fare` has 714 points piled up at low fares:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout='constrained')

sns.scatterplot(data=titanic, x='Age', y='Fare', ax=axes[0])
axes[0].set(title='Default: solid points hide each other')

sns.scatterplot(data=titanic, x='Age', y='Fare', alpha=0.3, edgecolor=None, ax=axes[1])
axes[1].set(yscale='log', title='alpha=0.3 + log scale: density shows through')
plt.show()

Transparency (`alpha`) makes dense areas darker, and the log scale (notebook 01) spreads out the crowded low fares.

## 2. Trend Lines with `sns.regplot()`
`regplot` draws a scatter plot plus a fitted straight line, with a shaded band showing how uncertain the line is:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(data=penguins, x='flipper_length_mm', y='body_mass_g',
            scatter_kws={'alpha': 0.4}, line_kws={'color': 'black'}, ax=ax)
ax.set(title='Body Mass Rises with Flipper Length', xlabel='Flipper length (mm)', ylabel='Body mass (g)')
plt.show()

## 3. 🤯 The Penguin Paradox, Made Visible
The Module 07 lab found that bill length and bill depth correlate **negatively** overall (r ≈ −0.24), but **positively within every species** (r ≈ +0.39 to +0.65). It took a table and a loop to find. Here it is in one figure:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True, layout='constrained')

# Left: everyone together, one line
sns.regplot(data=penguins, x='bill_length_mm', y='bill_depth_mm', ax=axes[0],
            scatter_kws={'alpha': 0.4, 'color': 'gray'}, line_kws={'color': 'black'})
r_all = penguins['bill_length_mm'].corr(penguins['bill_depth_mm'])
axes[0].set(title=f'All penguins together: r = {r_all:.2f}')

# Right: one line per species
palette = dict(zip(SPECIES_ORDER, sns.color_palette(n_colors=3)))
for species, group in penguins.groupby('species'):
    r = group['bill_length_mm'].corr(group['bill_depth_mm'])
    sns.regplot(data=group, x='bill_length_mm', y='bill_depth_mm', ax=axes[1],
                color=palette[species], scatter_kws={'alpha': 0.5}, label=f'{species} (r = {r:.2f})')
axes[1].set(title='Split by species: every trend reverses')
axes[1].legend()

for ax in axes:
    ax.set(xlabel='Bill length (mm)', ylabel='Bill depth (mm)')

fig.suptitle("Simpson's Paradox in the Palmer Penguins", fontsize=14)
fig.savefig(OUT / 'penguin_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

On the right, you can see **why**: the Gentoo cluster sits in the lower right (long, shallow bills), and the other two sit higher up. A single line through all three clusters slopes down, even though each cluster slopes up.

The left chart isn't "wrong" (it's the correct line for all the data combined), but it answers a question nobody meant to ask. **Whenever your data contain groups, color by group before you fit a line.**

## 4. Anscombe, One More Time
Notebook 01 showed the four Anscombe datasets. `sns.lmplot()` is the figure-level `regplot`: it fits a line in every panel:

In [ ]:
anscombe = pd.read_csv('anscombe.csv')

g = sns.lmplot(data=anscombe, x='x', y='y', col='dataset', col_wrap=2,
               height=3, aspect=1.2, ci=None, line_kws={'color': 'tab:red'})
g.figure.suptitle('The Same Fitted Line, Four Different Datasets', y=1.03)
plt.show()

The four red lines are **identical**: same slope, same intercept. A fitted line is another summary statistic, and it can hide just as much as a mean. Only dataset I is described well by its line. Look at the points before you trust the line.

✅ **Your Turn:** `regplot` can fit a curve with `order=2`. Redraw dataset II alone with `order=2`. Does the curve fit better than the straight line?

In [ ]:
# Your turn here

## 5. Categories: Counts, Estimates, and Error Bars
### `countplot`: how many rows in each category

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=titanic, x='class', hue='Sex', order=['1st', '2nd', '3rd'], ax=ax)
ax.set(title='Passengers by Class and Sex', xlabel='Ticket class', ylabel='Passengers')
plt.show()

### `barplot`: an **estimate** for each category
`sns.barplot()` computes a statistic of `y` for each group (the **mean**, by default) and draws an **error bar** on top. Since `Survived` is 0/1, the mean is the survival **rate** (Module 07 notebook 01):

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=titanic, x='class', y='Survived', hue='Sex', order=['1st', '2nd', '3rd'], ax=ax)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Survival Rate by Class and Sex (with 95% CI)', xlabel='Ticket class', ylabel='Survived', ylim=(0, 1))
plt.show()

Compare this with notebook 02's pandas version. You get the same bars, but Seaborn computed the rates from the raw rows (no `pivot_table` needed) and added error bars.

**What do the error bars mean?** By default, a 95% **confidence interval**: a range showing how much the estimate might change with a different sample of passengers. Narrow bars mean a precise estimate, usually from a big group. Seaborn's `errorbar=` changes what's drawn:

| `errorbar=` | Shows | Answers |
|---|---|---|
| `('ci', 95)` (default) | 95% confidence interval of the mean | "How sure am I about this group's average?" |
| `'se'` | ± one standard error | the same question, with a narrower band |
| `'sd'` | ± one standard deviation | "How spread out are the individual values?" (Module 07 notebook 02) |
| `None` | nothing | |

> ⚠️ Always say which one you drew. A reader can't tell an SD bar from a CI bar by looking. The CI uses random resampling, so its exact ends can shift slightly between runs. Pass `seed=` for repeatable results.

### `pointplot`: the same estimates, connected
A **point plot** draws each estimate as a dot. With `hue`, it connects the dots for each hue level, which makes differences in the **pattern** easy to compare:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.pointplot(data=titanic, x='class', y='Survived', hue='Sex', order=['1st', '2nd', '3rd'],
              dodge=0.2, ax=ax)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title='Survival Rate by Class and Sex', xlabel='Ticket class', ylabel='Survived', ylim=(0, 1))
plt.show()

The two lines aren't parallel. For women, the big drop is from 2nd to 3rd class. For men, it's from 1st to 2nd. When lines like these cross or diverge, the effect of one variable **depends on** the other (an **interaction**). Lines suit an ordered x-axis like class (Foundations 07). For an unordered one, prefer bars.

## 6. Heatmaps
A heatmap colors each cell of a table. Two tables from Module 07 are perfect for it.

### A correlation matrix
Correlations run from −1 to +1, with 0 meaning "nothing." That calls for a **diverging** color map, centered on 0. And because the matrix is symmetric (Module 07 notebook 05), mask the redundant upper triangle:

In [ ]:
corr = penguins[MEASUREMENTS].corr()
mask = np.triu(np.ones(corr.shape, dtype=bool))          # hide the diagonal and above

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Pearson r'}, ax=ax)
ax.set(title='Penguin Measurements: Correlations')
ax.grid(False)          # the whitegrid theme would draw lines through the masked cells
plt.show()

- `cmap='vlag'`: blue for negative, red for positive, and pale near zero.
- `vmin=-1, vmax=1, center=0`: fixes the color scale, so the same color always means the same r, in every heatmap you make.
- `annot=True, fmt='.2f'`: prints the numbers too. Color for the pattern, numbers for the detail.

### A pivot table of rates
Survival rates run from 0 to 1, with no meaningful middle. That calls for a **sequential** color map (light to dark):

In [ ]:
rates = titanic.pivot_table(values='Survived', index='class', columns='Sex', aggfunc='mean')

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(rates, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'Survival rate'}, ax=ax)
ax.set(title='Titanic Survival Rate', xlabel='', ylabel='Ticket class')
ax.grid(False)
plt.show()

| Data | Color map type | Examples |
|---|---|---|
| has a meaningful middle (correlation, change, difference) | **diverging** | `'vlag'`, `'coolwarm'`, `'RdBu'` |
| low to high, no middle (counts, rates, amounts) | **sequential** | `'Blues'`, `'viridis'`, `'rocket'` |
| unordered categories | **qualitative** | `'tab10'`, `'Set2'`, `'colorblind'` |

## 7. Every Pair at Once: `sns.pairplot()`
`pairplot` draws a scatter plot for every pair of numeric columns, with each column's distribution on the diagonal. It's the visual version of the full correlation matrix. `corner=True` skips the redundant upper half:

In [ ]:
g = sns.pairplot(penguins, vars=MEASUREMENTS, hue='species', hue_order=SPECIES_ORDER,
                 corner=True, height=2.2, plot_kws={'alpha': 0.5, 's': 20})
g.figure.suptitle('Every Pair of Penguin Measurements', y=1.02)
g.savefig(OUT / 'penguin_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

A pairplot is an **exploration** tool: fast, broad, and too busy to show anyone else. Use it to decide which two or three charts deserve to be made properly. (Which pair separates the three species most cleanly?)

---
### 📝 Summary
- `scatterplot` with `hue`/`style`/`size`. Fix overplotting with `alpha` and log scales.
- `regplot`/`lmplot` add fitted lines, and a line is a summary that can mislead (Anscombe).
- **Color by group before you fit a line**: one chart reveals Simpson's paradox.
- `countplot` counts rows, `barplot` estimates a statistic with an error bar, and `pointplot` connects estimates to show interactions. Say what your error bars mean.
- Heatmaps: diverging color maps (centered) for correlations, sequential ones for amounts. Fix `vmin`/`vmax`.
- `pairplot` for exploration, not presentation.

## 🏋️ Practice

### Practice 1 — Pokémon Scatter (easy)
Load `pokemon.csv`. Make a scatter plot of `Attack` vs. `Sp. Atk`, colored by `Legendary`, with transparency. Where do the Legendary Pokémon sit?

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here

### Practice 2 — A Heatmap of Pokémon Stats (medium)
Draw a masked, annotated correlation heatmap of the six Pokémon stats with a diverging color map fixed at −1 to 1. Which pair is strongest? Does that match your Module 07 notebook 05 results?

In [ ]:
# Your code here

### Practice 3 — Bars With Honest Error Bars (harder)
Draw mean `body_mass_g` by `species` twice, side by side: once with `errorbar=('ci', 95)` and once with `errorbar='sd'`. Use the same y-axis. In a Markdown cell, explain why the SD bars are so much longer, and which one you'd use to answer (a) "how precisely do we know the Gentoo average?" and (b) "how much do individual Gentoos vary?"

In [ ]:
# Your code here

*Your explanation here.*

## ✏️ Your Turn
Use `sns.lmplot()` with `hue='species'` and `col='sex'` to show the relationship between `flipper_length_mm` and `body_mass_g`. Save it as `OUT / 'flipper_mass_lmplot.png'`. Does the relationship look the same for every species and both sexes?

In [ ]:
# Your turn here

*Your answer here.*

## 🔥 Challenge (Optional) — Hunt for Another Paradox
The Module 07 lab's Practice 3 asked whether bill length/depth was the only pair that reverses within species. Answer it visually: use `sns.lmplot(..., hue='species')` on every pair of measurements (a loop over the pairs), plus a single `regplot` for the combined data. Which pairs, if any, have a combined slope with the opposite sign from all three species' slopes?

In [ ]:
# Your code here

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `04-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
✅ Next up → [Choosing and Telling](05-Choosing_and_Telling.ipynb): how to pick the right chart, keep it honest, and make it say one clear thing.